# RLS: таблица доступов `sbx_da.rls_acq_dashboard`

Собирает справочник прав для теста: вкладка **Эффективность ТСП** видна, но пустая
у пользователей с `allow_tsp_efficiency = 0`.

См. `HOW_TO_rls_tsp_efficiency_test.md`, `sources/sql/rls_acq_tsp_efficiency.sql`.

## Что делает
1. Готовит DataFrame доступов (логин → флаг).
2. DROP/CREATE `sbx_da.rls_acq_dashboard` в DRP.
3. GRANT SELECT → `raisa_superset`.

## Перед запуском
- Поправь список в config (логины **как в Superset** — проверь `{{ current_username() }}`).
- Для теста: `elina-ad` → `0`, себе → `1`.


In [ ]:
import getpass

import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

# --- config ---
drp_schema = 'sbx_da'
drp_table = 'rls_acq_dashboard'
target_fq = f'{drp_schema}.{drp_table}'
drp_superset_grant_role = 'raisa_superset'

# Логины = username в Superset (обычно без домена; сверь через Jinja current_username())
# allow_tsp_efficiency: '1' = видит данные эффективности, '0' = пустые чарты
ACL_ROWS = [
    {'username': 'Shestopalov-VYur', 'allow_tsp_efficiency': '1', 'comment': 'owner'},
    {'username': 'elina-ad', 'allow_tsp_efficiency': '0', 'comment': 'test: empty efficiency tab'},
    # {'username': 'other-user', 'allow_tsp_efficiency': '1', 'comment': ''},
]

acl_df = pd.DataFrame(ACL_ROWS)
acl_df['username'] = acl_df['username'].astype(str).str.strip()
acl_df['allow_tsp_efficiency'] = acl_df['allow_tsp_efficiency'].astype(str).str.strip()
acl_df['comment'] = acl_df['comment'].fillna('').astype(str)

# нормализация ключей
dup = acl_df['username'].str.lower().duplicated(keep=False)
if dup.any():
    raise RuntimeError('Дубли username (case-insensitive):\n' + acl_df.loc[dup].to_string())

print('target =', target_fq)
print('rows =', len(acl_df))
display(acl_df)


In [ ]:
drp_user = input('DRP user: ').strip()
drp_password = getpass.getpass('DRP password: ')

drp = connect(
    to='DRP',
    user_params={'user_name': drp_user, 'password': drp_password},
)
print('DRP connected as', drp_user)


In [ ]:
# Upload: DROP + CREATE + append (все колонки TEXT — как у остальных tmp_*)
upload_df = acl_df.copy()
for c in upload_df.columns:
    upload_df[c] = upload_df[c].map(lambda x: None if pd.isna(x) else str(x)).astype(object)

col_defs = [f'"{str(c)}" TEXT' for c in upload_df.columns]
create_sql = f'CREATE TABLE {target_fq} (\n  ' + ',\n  '.join(col_defs) + '\n)'

with drp:
    drp.execute(f'DROP TABLE IF EXISTS {target_fq}')
    drp.execute(create_sql)
    drp.write(table=target_fq, df=upload_df, mode='append')

    cnt = drp.fetch(f'SELECT COUNT(*) AS row_cnt FROM {target_fq}')
    preview = drp.fetch(
        f'''
        SELECT username, allow_tsp_efficiency, comment
        FROM {target_fq}
        ORDER BY username
        '''
    )

    try:
        drp.execute(f'GRANT USAGE ON SCHEMA {drp_schema} TO {drp_superset_grant_role}')
        drp.execute(f'GRANT SELECT ON TABLE {target_fq} TO {drp_superset_grant_role}')
        print(f'OK: GRANT SELECT ON {target_fq} TO {drp_superset_grant_role}')
    except Exception as grant_exc:
        print('WARNING: GRANT failed:', type(grant_exc).__name__, str(grant_exc)[:400])
        print(f'  GRANT SELECT ON TABLE {target_fq} TO {drp_superset_grant_role};')

print('OK DRP rows =', int(pd.to_numeric(cnt.iloc[0, 0], errors='coerce')))
display(preview)


In [ ]:
# Smoke: как отработает фильтр эффективности для логинов из ACL
smoke_sql = f'''
SELECT
  a.username,
  a.allow_tsp_efficiency,
  CASE
    WHEN BTRIM(CAST(a.allow_tsp_efficiency AS TEXT)) IN ('1', 'true', 'True', 'Y', 'y')
    THEN 'WILL_SEE_DATA'
    ELSE 'EMPTY_CHARTS'
  END AS expected_efficiency_tab
FROM {target_fq} a
ORDER BY a.username
'''
with drp:
    smoke = drp.fetch(smoke_sql)
display(smoke)

print('Next:')
print('1. Superset Virtual Dataset SQL from sources/sql/rls_acq_tsp_efficiency.sql (block 2)')
print('2. Jinja ON; charts на вкладке Эффективность → этот dataset')
print('3. Login as elina-ad → empty tables; your login → data OK')
print("4. Verify username in SQL Lab with Jinja: current_username()")
